Adverserial resistant Fake payment Gateway detection using transactions patterns
This project detects fake payment gateways using URL metadata, transaction patterns, and adversarial-resistant feature engineering.

**Data sets Collection (Without Adverserial)**
#
Adverserial is for fooling a model as attacker can be able to bypass

In [5]:
import pandas as pd
df = pd.read_csv('../raw/adverserial_legit.csv')
print(df)

                         url  label  TransactionAmount TransactionType  \
0          http://paypal.com      1               4.99           Debit   
1          http://google.com      0              50.00          Credit   
2        http://facebook.com      1               1.99           Debit   
3       http://microsoft.com      0             120.50           Debit   
4  https://bankofamerica.com      1               9.99           Debit   
5        http://linkedin.com      0               3.00          Credit   
6           http://chase.com      1               0.49           Debit   
7          http://amazon.com      0              75.00          Credit   

   CustomerAge  AccountBalance    Location  
0           35          1200.0      Sylhet  
1           45          5000.0       Dhaka  
2           22           500.0  Chittagong  
3           55         15000.0    Rajshahi  
4           60           300.0      Khulna  
5           30          7000.0       Dhaka  
6           19    

**Adverserial Generations**
#
interms of there homoglyph_domain,add_https,mimic_timing,age_spoof

In [2]:
import pandas as pd
import random
from urllib.parse import urlparse
import os


def homoglyph_domain(url):
    p = urlparse(url)
    host = p.netloc
    
    host2 = host.replace('l','I').replace('o','0').replace('a','@')
    return url.replace(host, host2)

def add_https(url):
    if url.startswith('http://'):
        return url.replace('http://', 'https://')
    return url

def mimic_timing(row):
    
    return random.uniform(5.0, 15.0)

def age_spoof(row):
    
    return random.choice([15, 30, 60, 90])


def generate_variants(df, n_variants=5):
    adv_rows = []
    
    
    phish_targets = df[df['label'] == 1]
    
    
    required_cols = list(df.columns) + ['notes', 'num_redirects', 'https_flag', 
                                       'domain_age_days', 'time_to_confirm', 'page_dwell']
    
    for _, row in phish_targets.iterrows():
        for i in range(n_variants):
            r = row.copy()
            
            for col in ['notes', 'num_redirects', 'https_flag', 'domain_age_days', 'time_to_confirm', 'page_dwell']:
                r[col] = None 
                
            attack = random.choice(['homoglyph','https','timing','age','redirects','amount_mimic'])
            
            if attack == 'homoglyph':
                r['url'] = homoglyph_domain(r['url'])
                r['notes'] = 'homoglyph'
            elif attack == 'https':
                r['url'] = add_https(r['url'])
                r['https_flag'] = 1
                r['notes'] = 'https_added'
            elif attack == 'timing':
                r['time_to_confirm'] = mimic_timing(row)
                r['page_dwell'] = max(0.5, random.uniform(0.7, 1.5)) 
                r['notes'] = 'timing_mimic'
            elif attack == 'age':
                r['domain_age_days'] = age_spoof(row)
                r['notes'] = 'age_spoof'
            elif attack == 'redirects':
                r['num_redirects'] = random.randint(3, 7) 
                r['notes'] = 'more_redirects'
            elif attack == 'amount_mimic':
                
                r['TransactionAmount'] = random.choice([0.49, 1.99, 5.0, 9.99])
                r['notes'] = 'amount_mimic'
            
            adv_rows.append(r)
            
   
    adv_df = pd.DataFrame(adv_rows)
   
    adv_df = adv_df.reindex(columns=required_cols)
    return adv_df.fillna('') 

if __name__ == "__main__":
    
    os.makedirs('../processed', exist_ok=True)
    os.makedirs('../raw', exist_ok=True)
    
    input_path = "../raw/adverserial_legit.csv"
    output_path = "../processed/sessions_with_adv.csv"
    
    try:
    
        manual_df = pd.read_csv(input_path)
    except FileNotFoundError:
        print(f"Error: {input_path} not found. Please create it with the updated structure first.")
        exit()
        
    print(f"Generating adversarial variants from {len(manual_df[manual_df['label'] == 1])} targets...")
    
    
    adv_test_df = generate_variants(manual_df)
    
    
    adv_test_df.to_csv(output_path, index=False)
    
    print(f"Saved {len(adv_test_df)} adversarial test samples to {output_path}")

Generating adversarial variants from 4 targets...
Saved 20 adversarial test samples to ../processed/sessions_with_adv.csv


**Dataset(Adverserial Generated)** 
#
Adverserial dataset created for model so that model understand adverserial datasets 


In [7]:
import pandas as pd
df = pd.read_csv('../processed/sessions_with_adv.csv')
print(df.head())

                  url  label  TransactionAmount TransactionType  CustomerAge  \
0   http://paypal.com      1               4.99           Debit           35   
1   http://p@yp@I.c0m      1               4.99           Debit           35   
2  https://paypal.com      1               4.99           Debit           35   
3   http://paypal.com      1               9.99           Debit           35   
4   http://paypal.com      1               5.00           Debit           35   

   AccountBalance Location           notes  num_redirects  https_flag  \
0          1200.0   Sylhet  more_redirects            7.0         NaN   
1          1200.0   Sylhet       homoglyph            NaN         NaN   
2          1200.0   Sylhet     https_added            NaN         1.0   
3          1200.0   Sylhet    amount_mimic            NaN         NaN   
4          1200.0   Sylhet    amount_mimic            NaN         NaN   

   domain_age_days  time_to_confirm  page_dwell  
0              NaN            

**Dataset(Transaction Patterns normal bank transaction data)**

In [9]:
import pandas as pd
df = pd.read_csv('../raw/bank_transactions_data_kaggle.csv')
print(df.head())

  TransactionID AccountID  TransactionAmount      TransactionDate  \
0      TX000001   AC00128              14.09  2023-04-11 16:29:14   
1      TX000002   AC00455             376.24  2023-06-27 16:44:19   
2      TX000003   AC00019             126.29  2023-07-10 18:16:08   
3      TX000004   AC00070             184.50  2023-05-05 16:32:11   
4      TX000005   AC00411              13.45  2023-10-16 17:51:24   

  TransactionType   Location DeviceID      IP Address MerchantID Channel  \
0           Debit  San Diego  D000380  162.198.218.92       M015     ATM   
1           Debit    Houston  D000051     13.149.61.4       M052     ATM   
2           Debit       Mesa  D000235  215.97.143.157       M009  Online   
3           Debit    Raleigh  D000187  200.13.225.150       M002  Online   
4          Credit    Atlanta  D000308    65.164.3.100       M091  Online   

   CustomerAge CustomerOccupation  TransactionDuration  LoginAttempts  \
0           70             Doctor                   81 

In [10]:
import pandas as pd
import random

input_path = "../raw/bank_transactions_data_kaggle.csv"
output_path = "../processed/kaggle_legit.csv"

bd_cities = ["Dhaka", "Chittagong", "Sylhet", "Rajshahi", "Khulna"]

print("Loading Kaggle dataset...")
df = pd.read_csv(input_path)

df_small = df.head(14)  

df_small["Location"] = [random.choice(bd_cities) for _ in range(len(df_small))]
df_small["label"] = 0  

df_small.to_csv(output_path, index=False)

print("Saved Kaggle legitimate dataset to data/raw/kaggle_legit.csv")


Loading Kaggle dataset...
Saved Kaggle legitimate dataset to data/raw/kaggle_legit.csv


C:\Users\Sadrib\AppData\Local\Temp\ipykernel_728\4290878812.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_small["Location"] = [random.choice(bd_cities) for _ in range(len(df_small))]
C:\Users\Sadrib\AppData\Local\Temp\ipykernel_728\4290878812.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_small["label"] = 0


Dataset(Legit Transactions Pattern with locations changed)

In [11]:
import pandas as pd
df = pd.read_csv('../processed/kaggle_legit.csv')
print(df.head())

  TransactionID AccountID  TransactionAmount      TransactionDate  \
0      TX000001   AC00128              14.09  2023-04-11 16:29:14   
1      TX000002   AC00455             376.24  2023-06-27 16:44:19   
2      TX000003   AC00019             126.29  2023-07-10 18:16:08   
3      TX000004   AC00070             184.50  2023-05-05 16:32:11   
4      TX000005   AC00411              13.45  2023-10-16 17:51:24   

  TransactionType    Location DeviceID      IP Address MerchantID Channel  \
0           Debit  Chittagong  D000380  162.198.218.92       M015     ATM   
1           Debit  Chittagong  D000051     13.149.61.4       M052     ATM   
2           Debit       Dhaka  D000235  215.97.143.157       M009  Online   
3           Debit  Chittagong  D000187  200.13.225.150       M002  Online   
4          Credit      Sylhet  D000308    65.164.3.100       M091  Online   

   CustomerAge CustomerOccupation  TransactionDuration  LoginAttempts  \
0           70             Doctor                

Datasets(Transactions patterns phishing using phishtank )

In [ ]:
import pandas as pd
import requests


url = "http://data.phishtank.com/data/online-valid.csv"

print("Downloading PhishTank dataset...")
df = pd.read_csv(url)

df[['url', 'phish_id']].to_csv("../data/raw/phishtank_urls.csv", index=False)

print("Saved to data/raw/phishtank_urls.csv")

**Data preprocessing** 
#
After we label all datasets This step merges the two collected datasets (kaggle_legit.csv and phishtank_urls.csv) with your manual adversarial data (adverserial_legit.csv) into a single clean DataFrame.
#

In [ ]:
import pandas as pd
import tldextract
import os

def extract_domain_features(url):

    ext = tldextract.extract(url)

    return ext.domain, ext.suffix

print("--- Running Preprocessing Pipeline ---")


try:
    phish = pd.read_csv("../raw/phishtank_urls.csv")
    legit = pd.read_csv("../processed/kaggle_legit.csv") 

    manual_data = pd.read_csv("../raw/adverserial_legit.csv") 
except FileNotFoundError as e:
    print(f"Error: Required file not found. Have you run collect_kaggle.py and collect_phishtank.py? Error: {e}")
    exit()


phish['label'] = 1  
phish_cols = ['url', 'label']


transaction_cols = ['TransactionAmount', 'TransactionType', 'CustomerAge', 'AccountBalance', 'Location']
for col in transaction_cols:
    phish[col] = None 
    
phish_df = phish[phish_cols + transaction_cols]


kaggle_df = legit[phish_cols + transaction_cols] 


manual_df = manual_data[phish_cols + transaction_cols]


df = pd.concat([
    phish_df, 
    kaggle_df,
    manual_df
], ignore_index=True).drop_duplicates(subset=['url']) 

print(f"Total training dataset size after merge: {len(df)}")

print("Extracting domain and TLD...")

df[['domain', 'tld']] = df['url'].apply(lambda u: pd.Series(extract_domain_features(u)))

output_path = "../processed/preprocessed_dataset.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True) # Ensure directory exists
df.to_csv(output_path, index=False)

print(f"Saved cleaned TRAINING dataset to {output_path}")

**Data processing Output**
# 
Merged and processed labelled data of transactions patterns, adverserial and fake detections

In [ ]:
import pandas as pd
df = pd.read_csv('../processed/preprocessed_dataset.csv')
print(df.head())


**Feature Engineering**
# 
This section creates advanced, robust features (like character entropy, domain length, etc.) from the combined dataset, preparing it for adverserial resistant and detecting fake payment detection approaches to the selected model.


In [ ]:
import pandas as pd
from urllib.parse import urlparse
import re
import numpy as np
from sklearn.preprocessing import LabelEncoder
import os

# Function to calculate character entropy (adversarial-resistant feature)
def calculate_entropy(s):
    if not isinstance(s, str) or not s:
        return 0.0
    p, lns = {}, 0.0
    for char in s:
        p[char] = p.get(char, 0) + 1
    for count in p.values():
        prob = count / len(s)
        lns += -prob * np.log2(prob) if prob > 0 else 0.0
    return lns

def create_engineered_features(df):
    
    print("Extracting URL Metadata Features...")
    
    df['url_length'] = df['url'].apply(len)
    df['path_length'] = df['url'].apply(lambda u: len(urlparse(u).path))
    
    
    ip_pattern = r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}'
    df['uses_ip'] = df['url'].apply(lambda u: 1 if re.search(ip_pattern, u) else 0)
    
    df['num_dots'] = df['url'].apply(lambda u: u.count('.'))
    
    df['uses_at_symbol'] = df['url'].apply(lambda u: 1 if '@' in u else 0)
    
    
    df['domain_entropy'] = df['domain'].apply(calculate_entropy)
    
    # 4. Transactional Feature Processing (Imputation and Encoding)
    print("Processing Transactional Features...")
    
    # Imputation: Fill missing numerical values (from PhishTank data) with the mean of the column
    numerical_cols = ['TransactionAmount', 'CustomerAge', 'AccountBalance']
    for col in numerical_cols:
        
        df[col] = df[col].fillna(df[col].mean()) 

    
    le = LabelEncoder()
    df['Location_Encoded'] = le.fit_transform(df['Location'].astype(str))
    
    # One-Hot Encoding for TransactionType (Debit/Credit is important)
    df = pd.get_dummies(df, columns=['TransactionType'], prefix='Txn')
    
    # Final cleanup: Drop the columns used for feature creation
    df_engineered = df.drop(columns=['url', 'domain', 'tld', 'Location', 'TransactionID', 'TransactionDate', 'TransactionDuration', 'DeviceID', 'IP Address', 'MerchantID', 'Channel', 'CustomerOccupation', 'LoginAttempts', 'PreviousTransactionDate'], errors='ignore')
    
   
    for col in df_engineered.columns:
        df_engineered[col] = pd.to_numeric(df_engineered[col], errors='coerce').fillna(0)
    
    return df_engineered

if __name__ == "__main__":
    
    input_path = "../processed/sessions_clean.csv"
    output_path = "../processed/sessions_engineered.csv"
    
    try:
        df = pd.read_csv(input_path)
    except FileNotFoundError:
        print("Error: sessions_clean.csv not found. Please run preprocess.py first.")
        exit()
    
    df_final = create_engineered_features(df.copy())
    

    df_final.to_csv(output_path, index=False)
    print(f"\nSaved final engineered dataset to {output_path}")